In [1]:
import csv
import random
import statistics
import sys; sys.path.append("..")
from task1_nqueens.min_conflicts import MinConflictsSolver, slow_conflicts

def run_experiments():
    N_values = [8, 16, 32, 64, 100, 200]
    seeds = list(range(1, 31))  # 30 seed co dinh
    max_steps = 20000
    modes = ["standard", "max_tabu", "roulette"]
    results = []

    print(f"{'N':>4} | {'Mode':>10} | {'Success (%)':>11} | {'Avg Steps':>10} | {'Avg Time (ms)':>13}")
    print("-" * 60)

    for N in N_values:
        for mode in modes:
            steps_list = []
            times_list = []
            success_cnt = 0
            tabu_len = max(2, min(10, N // 5))

            for s in seeds:
                # Tao cung 1 ban co ban dau cho moi cap (N, seed)
                init_rng = random.Random(s)
                init_state = [init_rng.randrange(N) for _ in range(N)]

                solver = MinConflictsSolver(N=N, max_steps=max_steps, mode=mode, tabu_size=tabu_len)
                res = solver.solve(initial_state=init_state, seed=s)

                if res["success"] and slow_conflicts(res["solution"]) == 0:
                    success_cnt += 1
                    steps_list.append(res["steps"])
                    times_list.append(res["time_sec"] * 1000.0)

            rate = (success_cnt / len(seeds)) * 100.0
            avg_steps = statistics.mean(steps_list) if steps_list else max_steps
            avg_time = statistics.mean(times_list) if times_list else 0.0

            row = {
                "N": N,
                "Mode": mode,
                "Success_Rate": round(rate, 2),
                "Avg_Steps": round(avg_steps, 2),
                "Avg_Time_ms": round(avg_time, 3),
            }
            results.append(row)
            print(f"{N:4d} | {mode:>10} | {rate:10.1f}% | {avg_steps:10.2f} | {avg_time:13.3f}")

    with open("t1_minconflicts_results.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["N", "Mode", "Success_Rate", "Avg_Steps", "Avg_Time_ms"])
        writer.writeheader()
        writer.writerows(results)

run_experiments()

   N |       Mode | Success (%) |  Avg Steps | Avg Time (ms)
------------------------------------------------------------
   8 |   standard |      100.0% |      43.63 |         0.470
   8 |   max_tabu |       86.7% |      16.15 |         0.263
   8 |   roulette |       93.3% |      59.61 |         0.896
  16 |   standard |       93.3% |     101.71 |         1.204
  16 |   max_tabu |      100.0% |      41.73 |         1.111
  16 |   roulette |      100.0% |      86.67 |         1.998
  32 |   standard |      100.0% |      84.03 |         3.297
  32 |   max_tabu |      100.0% |      49.90 |         1.075
  32 |   roulette |      100.0% |      87.03 |         2.626
  64 |   standard |      100.0% |     116.23 |         7.690
  64 |   max_tabu |      100.0% |      72.13 |         7.769
  64 |   roulette |      100.0% |     107.63 |         8.023
 100 |   standard |      100.0% |     136.63 |        20.294
 100 |   max_tabu |      100.0% |      92.20 |        16.790
 100 |   roulette |     